In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 03_global_tech_gold.data_cube;

In [0]:
from pyspark.sql.functions import col, lit, coalesce, when, concat_ws

In [0]:
# Load Dimension Tables
dim_date = spark.table("03_global_tech_gold.dims_tables.dim_date")
dim_company = spark.table("03_global_tech_gold.dims_tables.dim_company")
dim_department = spark.table("03_global_tech_gold.dims_tables.dim_department")
dim_employee = spark.table("03_global_tech_gold.dims_tables.dim_employee")
dim_account = spark.table("03_global_tech_gold.dims_tables.dim_account")

In [0]:
# Load Fact Tables
fact_general_ledger = spark.table("03_global_tech_gold.facts_tables.fact_general_ledger")
fact_payroll = spark.table("03_global_tech_gold.facts_tables.fact_payroll")

In [0]:
gl_data_cube = (
    fact_general_ledger.alias("f")
    .join(dim_date.alias("d_entry"), col("f.entry_date_key") == col("d_entry.date_key"), "left")
    .join(dim_date.alias("d_posting"), col("f.posting_date_key") == col("d_posting.date_key"), "left")
    .join(dim_company.alias("c"), col("f.company_key") == col("c.company_key"), "left")
    .join(dim_department.alias("dept"), col("f.department_key") == col("dept.department_key"), "left")
    .join(dim_account.alias("a"), col("f.account_key") == col("a.account_key"), "left")
    .select(
        
        col("f.gl_key"),
        col("f.gl_id"),
        col("f.reference_number"),
        col("f.description"),
        col("f.transaction_type"),
        col("f.gl_status"),
        
        col("f.entry_date"),
        col("d_entry.year").alias("entry_year"),
        col("d_entry.quarter").alias("entry_quarter"),
        col("d_entry.month").alias("entry_month"),
        col("d_entry.month_name").alias("entry_month_name"),
        col("d_entry.year_month").alias("entry_year_month"),
        
        col("f.posting_date"),
        col("d_posting.year").alias("posting_year"),
        col("d_posting.quarter").alias("posting_quarter"),
        col("d_posting.month").alias("posting_month"),
        col("d_posting.month_name").alias("posting_month_name"),
        col("f.year_month").alias("posting_year_month"),
        col("f.fiscal_year"),
        col("f.fiscal_period"),
        
        col("c.company_id"),
        col("c.company_name"),
        col("c.industry"),
        col("c.country"),
        col("c.company_is_active"),
        
        col("dept.department_id"),
        col("dept.department_code"),
        col("dept.department_name"),
        col("dept.cost_center"),
        
        col("a.account_id"),
        col("a.account_code"),
        col("a.account_name"),
        col("a.account_type"),
        col("a.category").alias("account_category"),
        col("a.account_is_active"),
        
        col("f.debit_amount"),
        col("f.credit_amount"),
        (col("f.debit_amount") - col("f.credit_amount")).alias("net_amount"),
        
        col("f.created_by"),
        col("f.approved_by")
    )
)

In [0]:
payroll_data_cube = (
    fact_payroll.alias("f")
    .join(dim_date.alias("d_pay"), col("f.pay_date_key") == col("d_pay.date_key"), "left")
    .join(dim_date.alias("d_start"), col("f.pay_period_start_date_key") == col("d_start.date_key"), "left")
    .join(dim_date.alias("d_end"), col("f.pay_period_end_date_key") == col("d_end.date_key"), "left")
    .join(dim_company.alias("c"), col("f.company_key") == col("c.company_key"), "left")
    .join(dim_department.alias("dept"), col("f.department_key") == col("dept.department_key"), "left")
    .join(dim_employee.alias("e"), col("f.employee_key") == col("e.employee_key"), "left")
    .select(
        
        col("f.payroll_key"),
        col("f.payroll_id"),
        col("f.payroll_status"),
        col("f.payment_method"),
        
        col("f.pay_date"),
        col("d_pay.year").alias("pay_year"),
        col("d_pay.quarter").alias("pay_quarter"),
        col("d_pay.month").alias("pay_month"),
        col("d_pay.month_name").alias("pay_month_name"),
        col("f.year_month").alias("pay_year_month"),
        
        col("f.pay_period_start"),
        col("f.pay_period_end"),
        
        col("c.company_id"),
        col("c.company_name"),
        col("c.industry"),
        col("c.country"),
        col("c.company_is_active"),
        
        col("dept.department_id"),
        col("dept.department_code"),
        col("dept.department_name"),
        col("dept.cost_center"),
        
        col("e.employee_id"),
        col("e.employee_code"),
        concat_ws(" ", col("e.first_name"), col("e.last_name")).alias("employee_name"),
        col("e.email"),
        col("e.position"),
        col("e.hire_date"),
        col("e.employee_is_active"),
        col("e.tenure_days"),
        
        col("f.bonus"),
        col("f.overtime_pay"),
        col("f.commission"),
        col("f.allowances"),
        col("f.total_compensation"),
        
        col("f.tax_deduction"),
        col("f.social_security"),
        col("f.health_insurance"),
        col("f.retirement_contribution"),
        col("f.other_deductions"),
        col("f.total_deduction"),
        
        col("f.net_salary")
    )
)

In [0]:
gl_data_cube.write.mode("overwrite").saveAsTable("03_global_tech_gold.data_cube.gl_data_cube")
payroll_data_cube.write.mode("overwrite").saveAsTable("03_global_tech_gold.data_cube.payroll_data_cube")


In [0]:
display(gl_data_cube)

In [0]:
from pyspark.sql.functions import sum as _sum, count, avg

gl_summary = (
    gl_data_cube
    .groupBy(
        "company_id", "company_name", "industry", "country",
        "department_id", "department_name", "cost_center",
        "posting_year_month", "posting_year", "posting_quarter", "posting_month",
        "account_category"
    )
    .agg(
        _sum("debit_amount").alias("total_debit"),
        _sum("credit_amount").alias("total_credit"),
        _sum("net_amount").alias("net_amount"),
        count("gl_id").alias("transaction_count")
    )
)

payroll_summary = (
    payroll_data_cube
    .groupBy(
        "company_id", "company_name", "industry", "country",
        "department_id", "department_name", "cost_center",
        "pay_year_month", "pay_year", "pay_quarter", "pay_month"
    )
    .agg(
        _sum("total_compensation").alias("total_compensation"),
        _sum("total_deduction").alias("total_deduction"),
        _sum("net_salary").alias("total_net_salary"),
        count("employee_id").alias("employee_count"),
        avg("net_salary").alias("avg_net_salary")
    )
)

unified_data_cube = (
    gl_summary.alias("gl")
    .join(
        payroll_summary.alias("pr"),
        [
            col("gl.company_id") == col("pr.company_id"),
            col("gl.department_id") == col("pr.department_id"),
            col("gl.posting_year_month") == col("pr.pay_year_month")
        ],
        "full_outer"
    )
    .select(
        coalesce(col("gl.company_id"), col("pr.company_id")).alias("company_id"),
        coalesce(col("gl.company_name"), col("pr.company_name")).alias("company_name"),
        coalesce(col("gl.industry"), col("pr.industry")).alias("industry"),
        coalesce(col("gl.country"), col("pr.country")).alias("country"),
        coalesce(col("gl.department_id"), col("pr.department_id")).alias("department_id"),
        coalesce(col("gl.department_name"), col("pr.department_name")).alias("department_name"),
        coalesce(col("gl.cost_center"), col("pr.cost_center")).alias("cost_center"),
        coalesce(col("gl.posting_year_month"), col("pr.pay_year_month")).alias("year_month"),
        coalesce(col("gl.posting_year"), col("pr.pay_year")).alias("year"),
        coalesce(col("gl.posting_quarter"), col("pr.pay_quarter")).alias("quarter"),
        coalesce(col("gl.posting_month"), col("pr.pay_month")).alias("month"),
        
        col("gl.account_category"),
        coalesce(col("gl.total_debit"), lit(0)).alias("total_debit"),
        coalesce(col("gl.total_credit"), lit(0)).alias("total_credit"),
        coalesce(col("gl.net_amount"), lit(0)).alias("gl_net_amount"),
        coalesce(col("gl.transaction_count"), lit(0)).alias("transaction_count"),
        
        coalesce(col("pr.total_compensation"), lit(0)).alias("total_compensation"),
        coalesce(col("pr.total_deduction"), lit(0)).alias("total_deduction"),
        coalesce(col("pr.total_net_salary"), lit(0)).alias("total_net_salary"),
        coalesce(col("pr.employee_count"), lit(0)).alias("employee_count"),
        col("pr.avg_net_salary")
    )
)

In [0]:
# Save Unified Data Cube
unified_data_cube.write.mode("overwrite").saveAsTable("03_global_tech_gold.data_cube.unified_data_cube")